# Can Small Language Models Review Research Papers? — full study (v3)

One run produces every number and figure in the paper:
* **Main study:** 200 ICLR 2024 submissions (100 accepted, 100 explicitly rejected) × 4 prompts × 4 models
  (Qwen2.5 1.5B / 3B / 7B and Phi-3.5-mini).
* **Contamination check:** the same on 200 ICLR 2025 submissions, published after these models were trained.
* **Analyses:** leniency, stability (50% and real-acceptance-rate cutoffs, pairwise flips, bootstrap CIs),
  agreement with humans, a human ceiling, simple text baselines, what the models track, multiple-comparison
  correction, power analysis, and a check of the expected-score method.

**Kaggle settings:** Accelerator = **GPU T4 x2**, Internet = **On**. Then **Save Version → Save & Run All (Commit)**.
Outputs (`data/`, `results/`) appear on the notebook page under **Output** when the run is complete.
If the run stops, run it again: finished scores are reused.

**On Google Colab:** Runtime → Change runtime type → **T4 GPU**. To resume, upload `scores_v3.csv` into a
folder named `results` (Files panel, left) before running. The next cell updates libraries (Colab only).

In [ ]:
import os, subprocess, sys
if not os.path.isdir("/kaggle"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "datasets",
                    "huggingface_hub", "statsmodels"], check=True)

## 1. Configuration

In [ ]:
import os, re, json, time, random, platform
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"   # report GPU errors at the exact operation (7B crashed on one paper in earlier runs)
import numpy as np
import pandas as pd

# Phi runs before Qwen 7B so its results finish even if 7B fails again
MODEL_IDS = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "microsoft/Phi-3.5-mini-instruct",
             "Qwen/Qwen2.5-7B-Instruct"]
BATCH_SIZE_BY_MODEL = {"Qwen/Qwen2.5-7B-Instruct": 1}   # one prompt at a time for 7B
DATASETS = {  # pinned dataset versions (Hugging Face commit hashes)
    2024: ("smallari/openreview-iclr2024-peer-reviews-RAW", "259c437a57a69bd13659daf56c60c678fbfc92cf"),
    2025: ("smallari/openreview-iclr2025-peer-reviews-RAW", "94e63f920fdbfad0cb25d5714dd60a8a2fc844fe"),
}
REAL_ACCEPT_RATE = {2024: 0.31, 2025: 0.32}   # ICLR's actual acceptance rates (secondary cutoff)
N_PER_CLASS = 100       # 100 accepted + 100 rejected per year
SEED = 42               # paper sampling
BOOT_SEED = 7           # bootstrap / permutation tests
N_BOOT = 1000
N_PERM = 2000
PASS_THRESHOLD = 6      # ICLR 6 = "marginally above the acceptance threshold"
BATCH_SIZE = 4

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
for d in ("data", "results"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
RUN_START = time.time()

# Resume: if an earlier run's output is attached as input, start from its finished scores
import glob, shutil
_prev = sorted(glob.glob("/kaggle/input/**/scores_v3.csv", recursive=True))
if _prev and not os.path.exists(f"{OUT}/results/scores_v3.csv"):
    shutil.copy(_prev[0], f"{OUT}/results/scores_v3.csv")
    print("resumed from", _prev[0], "-", len(pd.read_csv(_prev[0])), "finished scores")

## 2. Data
Accepted = any decision starting with "Accept". Rejected = decision exactly "Reject".
Papers with **no decision** (withdrawn or desk-rejected) are **excluded**; earlier versions wrongly counted them as rejected.

In [ ]:
from datasets import load_dataset

def human_ratings(reviews):
    """Rating -> int. 2024 stores '6: marginally above the acceptance threshold', 2025 stores 6."""
    out = []
    for r in (reviews if reviews is not None else []):
        v = r.get("rating")
        if isinstance(v, (int, float, np.integer, np.floating)) and not pd.isna(v):
            out.append(int(v))
        elif isinstance(v, str):
            m = re.match(r"\s*(\d+)", v)
            if m:
                out.append(int(m.group(1)))
    return out

data_log, papers_by_year = {}, {}
for year, (ds_id, rev) in DATASETS.items():
    raw = load_dataset(ds_id, split="raw", revision=rev).to_pandas()
    raw = raw[raw["year"].astype(str) == str(year)].drop_duplicates("paper_id").copy()
    raw["decision"] = raw["decision"].fillna("")
    raw["ratings"] = raw["reviews"].apply(human_ratings)
    raw["accepted"] = raw["decision"].str.startswith("Accept")
    raw["rejected"] = raw["decision"] == "Reject"
    keep = (raw["accepted"] | raw["rejected"]) & raw["abstract"].fillna("").str.strip().astype(bool) \
           & (raw["ratings"].str.len() >= 3)
    df = raw[keep].copy()
    df["label"] = df["accepted"].astype(int)
    df["n_reviews"] = df["ratings"].str.len()
    df["human_mean"] = df["ratings"].apply(np.mean)
    df["human_std"] = df["ratings"].apply(np.std)
    papers = pd.concat([df[df["label"] == lab].sample(N_PER_CLASS, random_state=SEED) for lab in (1, 0)])
    papers = papers[["paper_id", "title", "abstract", "decision", "label", "n_reviews",
                     "human_mean", "human_std", "ratings"]].reset_index(drop=True)
    papers["year"] = year
    papers.to_csv(f"{OUT}/data/papers_{year}.csv", index=False)
    papers_by_year[year] = papers
    data_log[year] = {
        "dataset": ds_id, "revision": rev, "rows": int(len(raw)),
        "decision_counts": raw["decision"].replace("", "(none: withdrawn/desk-rejected)").value_counts().to_dict(),
        "eligible_accepted": int((df["label"] == 1).sum()), "eligible_rejected": int((df["label"] == 0).sum()),
        "sample_mean_rating": papers.groupby("label")["human_mean"].mean().round(2).to_dict(),
        "sample_mean_reviews": round(float(papers["n_reviews"].mean()), 2),
    }
    print(year, json.dumps(data_log[year], indent=1, default=str))

with open(f"{OUT}/data/paper_ids.txt", "w") as f:
    for year, p in papers_by_year.items():
        for pid, lab in zip(p["paper_id"], p["label"]):
            f.write(f"{year}\t{pid}\t{'accepted' if lab else 'rejected'}\n")

## 3. Prompts (unchanged from v1/v2)

In [ ]:
PROMPTS = {
    "P1_expert": ("You are an expert reviewer for ICLR. Read the title and abstract below and rate the paper's "
                  "overall quality on a scale from 1 (strong reject) to 10 (strong accept). "
                  "Respond with a single integer only.\n\nTitle: {title}\n\nAbstract: {abstract}"),
    "P2_plain": ("Below is a submission to the ICLR conference. As a reviewer, score it from 1 to 10, where 10 means "
                 "a clear accept and 1 means a clear reject. Output just the number.\n\n"
                 "Title: {title}\nAbstract: {abstract}"),
    "P3_after": ("Title: {title}\n\nAbstract: {abstract}\n\nYou are reviewing this paper for ICLR, a top machine "
                 "learning conference. On a 1-10 scale (1 = strong reject, 10 = strong accept), what score would you "
                 "give? Reply with one integer and nothing else."),
    "P4_terse": ("Rate this ICLR submission.\nScale: 1 (definitely reject) to 10 (definitely accept).\n"
                 "Format: a single integer.\n\n### Title\n{title}\n\n### Abstract\n{abstract}"),
}

## 4. Scoring
* **Written score:** the integer the model writes (greedy decoding, ≤ 8 new tokens).
* **Expected score:** Σ s·P(s) over s = 1…10 from the model's probabilities for its first answer token.
  Works for any tokenizer: if "10" is two tokens ("1","0"), a second pass splits P("1") into 1 vs 10.
  All ten probabilities are saved (p1…p10) so others can recompute.
* The chat template is applied with **no system message**, exactly as in v1/v2. For Qwen this means the
  template inserts its default system prompt ("You are Qwen, created by Alibaba Cloud. You are a helpful assistant.").

In [ ]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import model_info

SCORES_CSV = f"{OUT}/results/scores_v3.csv"

def parse_score(text):
    m = re.search(r"\b(10|[1-9])\b", text)
    return int(m.group(1)) if m else np.nan

def answer_token_ids(tok, prompt_text):
    """Token ids that can start each answer 1-9 right after the prompt, how '10' is tokenized, and any
    whitespace-only token the tokenizer puts before the digit (Phi/Llama-style tokenizers emit '▁' first)."""
    base = tok(prompt_text).input_ids
    prefix = []
    def new_tokens(s):
        full = tok(prompt_text + s).input_ids
        if full[:len(base)] != base:
            return None
        nt = full[len(base):]
        lead = []
        while nt and tok.decode([nt[0]]).strip() == "":   # leading whitespace-only token(s)
            lead.append(nt[0])
            nt = nt[1:]
        if s == "1":
            prefix.extend(lead)
        return nt
    first = {}
    for d in range(1, 10):
        ids = set()
        nt = new_tokens(str(d))
        if nt:
            ids.add(nt[0])
        for v in (str(d), "▁" + str(d), "Ġ" + str(d)):
            i = tok.convert_tokens_to_ids(v)
            if isinstance(i, int) and i != tok.unk_token_id:
                ids.add(i)
        first[d] = sorted(ids)
    ten = new_tokens("10")
    return first, ten, prefix

def last_probs(model, input_ids, attention_mask):
    pos = (attention_mask.cumsum(-1) - 1).clamp(min=0)   # correct positions under left padding
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask, position_ids=pos).logits[:, -1, :]
    return logits.float().softmax(-1)

def append_token(ids, mask, token_id):
    t = torch.full((ids.shape[0], 1), token_id, device=ids.device)
    return torch.cat([ids, t], 1), torch.cat([mask, torch.ones_like(t)], 1)

def score_probs(model, enc, first, ten, prefix=()):
    """Returns (n, 10) array of P(answer = 1..10), renormalized, and the raw numeric mass
    (= P(prefix) x P(any digit answer)); mass near 1 means the model really answers with a number here."""
    ids, mask = enc["input_ids"], enc["attention_mask"]
    p_prefix = torch.ones(ids.shape[0], device=ids.device)
    for t in prefix:                                                       # e.g. Phi writes '▁' before the digit
        p_prefix = p_prefix * last_probs(model, ids, mask)[:, t]
        ids, mask = append_token(ids, mask, t)
    p = last_probs(model, ids, mask)
    P = torch.stack([p[:, first[d]].sum(1) for d in range(1, 10)], 1)        # P(first digit token starts d)
    p10 = torch.zeros_like(P[:, 0])
    if ten is not None and len(ten) == 1 and ten[0] not in first[1]:      # "10" is a single token
        p10 = p[:, ten[0]]
    elif ten is not None and len(ten) == 2:                                # "10" = "1" + "0": split P("1")
        q = last_probs(model, *append_token(ids, mask, ten[0]))[:, ten[1]]
        p10 = P[:, 0] * q
        P[:, 0] = P[:, 0] * (1 - q)
    P = torch.cat([P, p10[:, None]], 1)
    mass = P.sum(1)
    return (P / mass[:, None]).cpu().numpy(), (p_prefix * mass).cpu().numpy()

done = pd.read_csv(SCORES_CSV) if os.path.exists(SCORES_CSV) else pd.DataFrame(columns=["model", "year", "paper_id", "prompt_id"])
done_keys = set(zip(done["model"], done["year"].astype(int), done["paper_id"], done["prompt_id"]))
run_log = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
           "n_gpus": torch.cuda.device_count(), "dtype": "float16", "models": {}}

for model_id in MODEL_IDS:
    jobs = [(year, p.paper_id, pid, tmpl.format(title=p.title.strip(), abstract=p.abstract.strip()))
            for year, papers in papers_by_year.items() for p in papers.itertuples()
            for pid, tmpl in PROMPTS.items() if (model_id, year, p.paper_id, pid) not in done_keys]
    info = model_info(model_id)
    run_log["models"][model_id] = {"revision": info.sha}
    if not jobs:
        print("already done:", model_id)
        continue
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_id, revision=info.sha, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    load_kw = {}
    if torch.cuda.device_count() == 1 and "7B" in model_id:
        # single 16 GB GPU (P100): keep ~3 GB free for activations, put the remaining layers in CPU RAM
        load_kw["max_memory"] = {0: "13GiB", "cpu": "28GiB"}
    model = AutoModelForCausalLM.from_pretrained(model_id, revision=info.sha, dtype=torch.float16,
                                                 device_map="auto", **load_kw).eval()
    probe = tok.apply_chat_template([{"role": "user", "content": "x"}], tokenize=False, add_generation_prompt=True)
    first, ten, prefix = answer_token_ids(tok, probe)
    run_log["models"][model_id].update({"answer_first_token_ids": first, "ten_tokens": ten, "answer_prefix_tokens": prefix})
    print(model_id, "| '10' tokenized as", ten, "| prefix", prefix)
    bs = BATCH_SIZE_BY_MODEL.get(model_id, BATCH_SIZE)
    for i in range(0, len(jobs), bs):
        batch = jobs[i:i + bs]
        with open(f"{OUT}/results/last_job.txt", "w") as f:   # if the GPU crashes, this names the exact prompt
            f.write("\n".join(f"{model_id}\t{y}\t{pid}\t{pr}\tchars={len(c)}" for y, pid, pr, c in batch))
        texts = [tok.apply_chat_template([{"role": "user", "content": c}], tokenize=False, add_generation_prompt=True)
                 for *_, c in batch]
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=8, do_sample=False, temperature=None, top_p=None, top_k=None,
                                 pad_token_id=tok.pad_token_id)
        outs = tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        P, mass = score_probs(model, enc, first, ten, prefix)
        exp = P @ np.arange(1, 11)
        rows = []
        for (year, pid, pr, _), o, e, m, pr_row in zip(batch, outs, exp, mass, P):
            row = {"model": model_id, "year": year, "paper_id": pid, "prompt_id": pr, "raw_output": o.strip(),
                   "score": parse_score(o), "exp_score": e, "digit_mass": m}
            row.update({f"p{s}": pr_row[s - 1] for s in range(1, 11)})
            rows.append(row)
        pd.DataFrame(rows).to_csv(SCORES_CSV, mode="a", header=not os.path.exists(SCORES_CSV), index=False)
        print(f"{model_id}: {min(i + bs, len(jobs))}/{len(jobs)}", end="\r")
    run_log["models"][model_id]["minutes"] = round((time.time() - t0) / 60, 1)
    print(f"\n{model_id} done in {run_log['models'][model_id]['minutes']} min")
    del model
    torch.cuda.empty_cache()

## 5. Sanity checks
Targets: parse rate ≥ 0.95; numeric probability mass close to 1; written score usually equals the most likely score.

In [ ]:
scores = pd.read_csv(SCORES_CSV)
scores["year"] = scores["year"].astype(int)
pcols = [f"p{s}" for s in range(1, 11)]
scores["argmax_score"] = scores[pcols].values.argmax(1) + 1
checks = scores.groupby(["year", "model"]).agg(
    rows=("score", "size"), parse_rate=("score", lambda s: s.notna().mean()), digit_mass=("digit_mass", "mean"),
    written_equals_most_likely=("score", lambda s: (s == scores.loc[s.index, "argmax_score"]).mean()))
print(checks.round(3).to_string())
# Validation of the "1 vs 10" split: when the model actually wrote 10, how much probability went to 10?
w10 = scores[scores["score"] == 10]
ten_check = {"n_written_10": int(len(w10)), "mean_p10_when_written_10": float(w10["p10"].mean()) if len(w10) else None,
             "mean_p1_when_written_10": float(w10["p1"].mean()) if len(w10) else None}
print("10-check:", ten_check)
print(pd.crosstab([scores["year"], scores["model"]], scores["score"], normalize="index").round(2).to_string())

## 6. Analysis

In [ ]:
from scipy.stats import spearmanr, norm
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(BOOT_SEED)
rho = lambda x, y: spearmanr(x, y)[0]
auc = lambda x, y: roc_auc_score(y, x)
short = lambda m: m.split("/")[-1].replace("-Instruct", "").replace("-instruct", "")
HYPE = ["novel", "state-of-the-art", "state of the art", "sota", "significantly", "significant", "outperform",
        "outperforms", "superior", "remarkable", "unprecedented", "groundbreaking", "first", "breakthrough",
        "substantially", "dramatically", "promising", "effective", "efficient", "robust"]

def boot(fn, n, *arrays):
    """95% bootstrap interval of fn over resampled paper indices."""
    vals = []
    for _ in range(N_BOOT):
        i = rng.integers(0, n, n)
        try:
            v = fn(*[a[i] for a in arrays])
        except ValueError:
            continue
        if v is not None and not np.isnan(v):
            vals.append(v)
    return np.percentile(vals, [2.5, 97.5]).tolist()

def perm_p(x, y, stat):
    """Two-sided permutation p-value for stat(x, y) against no relationship."""
    obs = abs(stat(x, y) - (0.5 if stat is auc else 0))
    cnt = sum(abs(stat(x, rng.permutation(y)) - (0.5 if stat is auc else 0)) >= obs for _ in range(N_PERM))
    return (cnt + 1) / (N_PERM + 1)

def flip_rate(W, frac):
    """Share of papers whose accept verdict (top `frac` per prompt) differs across prompts."""
    k = int(round(frac * len(W)))
    top = np.zeros_like(W, dtype=bool)
    for j in range(W.shape[1]):
        top[np.argsort(-W[:, j], kind="stable")[:k], j] = True
    return (top.any(1) & ~top.all(1)).mean()

def level_shift(W):
    m = W.mean(0)
    return m.max() - m.min()

def text_features(papers):
    ab = papers["abstract"].fillna("").str.lower()
    words = ab.str.split().str.len()
    hype = ab.apply(lambda t: sum(len(re.findall(r"\b" + re.escape(h) + r"\b", t)) for h in HYPE))
    return words.values.astype(float), hype.values.astype(float), (hype / words * 100).values

def tfidf_auc(papers):
    """Cross-validated AUC of a bag-of-words classifier (10 × 5-fold)."""
    text = (papers["title"] + ". " + papers["abstract"]).values
    y = papers["label"].values
    aucs = []
    for r in range(10):
        skf = StratifiedKFold(5, shuffle=True, random_state=r)
        clf = make_pipeline(TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True),
                            LogisticRegression(C=1.0, max_iter=2000))   # vocabulary fit inside each fold
        pr = cross_val_predict(clf, text, y, cv=skf, method="predict_proba")[:, 1]
        aucs.append(roc_auc_score(y, pr))
    return float(np.mean(aucs)), [float(np.min(aucs)), float(np.max(aucs))]

def human_ceiling(papers, model_scores=None, draws=N_BOOT):
    """Leave-one-reviewer-out: one random reviewer vs. the mean of the others.
    If model_scores is given, the model is compared against the same 'others' mean (fair comparison)."""
    R = papers["ratings"].tolist()
    hum, mod = [], []
    for _ in range(draws):
        one, rest = [], []
        for r in R:
            j = rng.integers(0, len(r))
            one.append(r[j]); rest.append(np.mean([v for i, v in enumerate(r) if i != j]))
        hum.append(rho(one, rest))
        if model_scores is not None:
            mod.append(rho(model_scores, rest))
    out = {"human_single_vs_others": float(np.mean(hum)), "human_ci": np.percentile(hum, [2.5, 97.5]).tolist()}
    if model_scores is not None:
        out["model_vs_others"] = float(np.mean(mod))
    return out

def power_needed(r, power=0.8, alpha=0.05):
    return int(np.ceil(((norm.ppf(1 - alpha / 2) + norm.ppf(power)) / np.arctanh(r)) ** 2 + 3))

def power_at(r, n, alpha=0.05):
    z = np.arctanh(r) * np.sqrt(n - 3)
    return float(norm.cdf(z - norm.ppf(1 - alpha / 2)) + norm.cdf(-z - norm.ppf(1 - alpha / 2)))

results = {"data": data_log, "run": run_log, "sanity": json.loads(checks.round(4).reset_index().to_json(orient="records")),
           "ten_check": ten_check, "years": {}}

for year, papers in papers_by_year.items():
    meta = papers.set_index("paper_id")
    papers["ratings"] = papers["ratings"].apply(lambda r: list(r) if not isinstance(r, str) else json.loads(r))
    words, hype, hype_rate = text_features(papers)
    yr = {"n_papers": len(papers), "models": {}, "baselines": {}, "power": {}}
    y_lab, y_hum = papers["label"].values, papers["human_mean"].values
    # --- human ceiling & baselines ---
    yr["human_ceiling"] = human_ceiling(papers)
    for name, feat in [("abstract_length", words), ("hype_words", hype), ("hype_per_100_words", hype_rate)]:
        yr["baselines"][name] = {"spearman_vs_human": rho(feat, y_hum), "auc": auc(feat, y_lab),
                                 "auc_ci": boot(auc, len(feat), feat, y_lab)}
    m_auc, rng_auc = tfidf_auc(papers)
    yr["baselines"]["tfidf_logreg_cv"] = {"auc": m_auc, "auc_range_over_10_repeats": rng_auc}
    n = len(papers)
    yr["power"] = {"n": n, "power_at_r0.2": power_at(0.2, n), "power_at_r0.3": power_at(0.3, n),
                   "n_for_80pct_r0.2": power_needed(0.2), "n_for_80pct_r0.3": power_needed(0.3)}
    pvals, plabels = [], []
    for model_id in MODEL_IDS:
        s = scores[(scores["model"] == model_id) & (scores["year"] == year)]
        wide = s.pivot_table(index="paper_id", columns="prompt_id", values="exp_score").reindex(papers["paper_id"])
        greedy = s.pivot_table(index="paper_id", columns="prompt_id", values="score").reindex(papers["paper_id"])
        ok = wide.notna().all(1).values
        W, G = wide.values[ok], greedy.values[ok]
        yl, yh = y_lab[ok], y_hum[ok]
        Z = (W - W.mean(0)) / W.std(0)                          # z-score within prompt: removes level shifts
        avg = W.mean(1)
        cols = list(wide.columns)
        gv = G[~np.isnan(G)].ravel()
        mode = pd.Series(gv).mode()[0]
        pair_rho = {f"{cols[a]}~{cols[b]}": rho(W[:, a], W[:, b]) for a in range(4) for b in range(a + 1, 4)}
        pair_flip50 = {f"{cols[a]}~{cols[b]}": flip_rate(W[:, [a, b]], 0.5) for a in range(4) for b in range(a + 1, 4)}
        m = {
            "n": int(ok.sum()),
            "leniency": {"written_mode": float(mode), "written_mode_share": float((gv == mode).mean()),
                         "share_written_ge6": float((gv >= PASS_THRESHOLD).mean())},
            "stability": {
                "flip_rate_50": flip_rate(W, 0.5), "flip_rate_50_ci": boot(lambda w: flip_rate(w, 0.5), len(W), W),
                "flip_rate_50_random": 1 - 2 * 0.5 ** 4,
                "flip_rate_real": flip_rate(W, REAL_ACCEPT_RATE[year]),
                "flip_rate_real_ci": boot(lambda w: flip_rate(w, REAL_ACCEPT_RATE[year]), len(W), W),
                "flip_rate_real_random": 1 - REAL_ACCEPT_RATE[year] ** 4 - (1 - REAL_ACCEPT_RATE[year]) ** 4,
                "pairwise_flip_50": pair_flip50, "pairwise_flip_50_random": 0.5,
                "pairwise_spearman": pair_rho, "pairwise_spearman_mean": float(np.mean(list(pair_rho.values()))),
                "pairwise_spearman_min": float(np.min(list(pair_rho.values()))),
                "prompt_means": dict(zip(cols, W.mean(0).round(3).tolist())),
                "level_shift": level_shift(W), "level_shift_ci": boot(level_shift, len(W), W),
                "per_paper_std_raw": float(W.std(1).mean()), "per_paper_std_zscored": float(Z.std(1).mean()),
            },
            "agreement": {},
            "tracks": {"spearman_with_abstract_length": rho(avg, words[ok]), "spearman_with_hype_words": rho(avg, hype[ok]),
                       "spearman_with_hype_rate": rho(avg, hype_rate[ok])},
            "ceiling_comparison": human_ceiling(papers[ok], model_scores=avg, draws=300),
        }
        for j, p in enumerate(cols + ["average"]):
            x = avg if p == "average" else W[:, j]
            a = {"spearman": rho(x, yh), "spearman_ci": boot(rho, len(x), x, yh), "spearman_p": perm_p(x, yh, rho),
                 "auc": auc(x, yl), "auc_ci": boot(auc, len(x), x, yl), "auc_p": perm_p(x, yl, auc)}
            m["agreement"][p] = a
            pvals += [a["spearman_p"], a["auc_p"]]
            plabels += [(model_id, p, "spearman"), (model_id, p, "auc")]
        yr["models"][model_id] = m
    # Holm correction across every agreement test in this year
    rej, padj, *_ = multipletests(pvals, method="holm")
    for (model_id, p, kind), pa, r in zip(plabels, padj, rej):
        yr["models"][model_id]["agreement"][p][f"{kind}_p_holm"] = float(pa)
        yr["models"][model_id]["agreement"][p][f"{kind}_significant_holm"] = bool(r)
    results["years"][year] = yr

results["run"]["total_minutes"] = round((time.time() - RUN_START) / 60, 1)
with open(f"{OUT}/results/metrics_v3.json", "w") as f:
    json.dump(results, f, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print(json.dumps({y: {k: v for k, v in r.items() if k != "models"} for y, r in results["years"].items()}, indent=1, default=float))

## 7. Tables (markdown, pasted into the paper)

In [ ]:
def f2(v, ci=None):
    return f"{v:.2f}" + (f" [{ci[0]:.2f}, {ci[1]:.2f}]" if ci else "")

lines = []
for year, yr in results["years"].items():
    lines += [f"## ICLR {year} (n = {yr['n_papers']})", "",
              "**Table A. Leniency and stability**", "",
              "| Model | Most common written score (share) | Written ≥6 | Mean pairwise Spearman between prompts (min) | "
              "Flip rate @50% [95% CI] (random) | Flip rate @real rate [95% CI] (random) | Mean-score shift across prompts [95% CI] |",
              "|---|---|---|---|---|---|---|"]
    for model_id, m in yr["models"].items():
        L, S = m["leniency"], m["stability"]
        lines.append(f"| {short(model_id)} | {L['written_mode']:.0f} ({L['written_mode_share']:.0%}) | {L['share_written_ge6']:.0%} | "
                     f"{S['pairwise_spearman_mean']:.2f} ({S['pairwise_spearman_min']:.2f}) | "
                     f"{S['flip_rate_50']:.0%} [{S['flip_rate_50_ci'][0]:.0%}, {S['flip_rate_50_ci'][1]:.0%}] ({S['flip_rate_50_random']:.0%}) | "
                     f"{S['flip_rate_real']:.0%} [{S['flip_rate_real_ci'][0]:.0%}, {S['flip_rate_real_ci'][1]:.0%}] ({S['flip_rate_real_random']:.0%}) | "
                     f"{f2(S['level_shift'], S['level_shift_ci'])} |")
    lines += ["", "**Table B. Pairwise flip rates @50% (random = 50%)**", "",
              "| Model | " + " | ".join(next(iter(yr["models"].values()))["stability"]["pairwise_flip_50"]) + " |",
              "|---|" + "---|" * 6]
    for model_id, m in yr["models"].items():
        lines.append(f"| {short(model_id)} | " + " | ".join(f"{v:.0%}" for v in m["stability"]["pairwise_flip_50"].values()) + " |")
    hc = yr["human_ceiling"]
    lines += ["", "**Table C. Agreement with humans** (Spearman vs mean human rating; AUC accepted vs rejected; "
              "95% CI; * = significant after Holm correction)", "",
              f"Human ceiling (one reviewer vs mean of the others): Spearman {f2(hc['human_single_vs_others'], hc['human_ci'])}", "",
              "| Model | Prompt | Spearman [95% CI] | AUC [95% CI] |", "|---|---|---|---|"]
    for model_id, m in yr["models"].items():
        for p, a in m["agreement"].items():
            st = lambda k: "*" if a[f"{k}_significant_holm"] else ""
            lines.append(f"| {short(model_id)} | {p} | {f2(a['spearman'], a['spearman_ci'])}{st('spearman')} | "
                         f"{f2(a['auc'], a['auc_ci'])}{st('auc')} |")
    lines += ["", "**Table D. Same comparison as the human ceiling** (vs mean of the *other* reviewers, prompt-averaged score)", "",
              "| Model | Model vs others | Human single reviewer vs others |", "|---|---|---|"]
    for model_id, m in yr["models"].items():
        c = m["ceiling_comparison"]
        lines.append(f"| {short(model_id)} | {c['model_vs_others']:.2f} | {c['human_single_vs_others']:.2f} |")
    lines += ["", "**Table E. Simple baselines (no LLM)**", "", "| Baseline | Spearman vs human | AUC |", "|---|---|---|"]
    for name, b in yr["baselines"].items():
        sp = f"{b['spearman_vs_human']:.2f}" if "spearman_vs_human" in b else "—"
        ci = b.get("auc_ci") or b.get("auc_range_over_10_repeats")
        lines.append(f"| {name} | {sp} | {f2(b['auc'], ci)} |")
    lines += ["", "**Table F. What do model scores track?** (Spearman of prompt-averaged score with abstract features)", "",
              "| Model | Abstract length | Hype words | Hype per 100 words |", "|---|---|---|---|"]
    for model_id, m in yr["models"].items():
        t = m["tracks"]
        lines.append(f"| {short(model_id)} | {t['spearman_with_abstract_length']:.2f} | {t['spearman_with_hype_words']:.2f} | "
                     f"{t['spearman_with_hype_rate']:.2f} |")
    pw = yr["power"]
    lines += ["", f"Power: with n = {pw['n']}, power to detect r = 0.2 is {pw['power_at_r0.2']:.0%}, r = 0.3 is {pw['power_at_r0.3']:.0%}. "
              f"80% power needs n = {pw['n_for_80pct_r0.2']} (r = 0.2) or {pw['n_for_80pct_r0.3']} (r = 0.3).", ""]
lines += ["## Run", "", "```", json.dumps(results["run"], indent=1, default=str), "```", "",
          f"10-check: {ten_check}"]
table = "\n".join(lines)
with open(f"{OUT}/results/tables_v3.md", "w") as f:
    f.write(table + "\n")
print(table)

## 8. Figures (PNG for README, PDF for the paper)

In [ ]:
import matplotlib.pyplot as plt

COLORS = dict(zip(PROMPTS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))   # colorblind-validated categorical order
MARKERS = dict(zip(PROMPTS, ["o", "s", "^", "D"]))                           # shape = second encoding
INK, MUTED, GRID = "#1f1f1e", "#6b6a63", "#e6e5e0"
plt.rcParams.update({"font.size": 9, "text.color": INK, "axes.labelcolor": INK, "axes.edgecolor": MUTED,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True})

def save(fig, name):
    fig.savefig(f"{OUT}/results/{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{OUT}/results/{name}.pdf", bbox_inches="tight")
    plt.show()

# Figure 1 (ICLR 2024): expected scores under each prompt + the human mean rating, same 1-10 scale
papers = papers_by_year[2024].sort_values("human_mean").reset_index(drop=True)
xs = np.arange(len(papers))
fig, axes = plt.subplots(len(MODEL_IDS), 1, figsize=(7.5, 2.1 * len(MODEL_IDS)), sharex=True, squeeze=False)
for ax, model_id in zip(axes[:, 0], MODEL_IDS):
    s = scores[(scores["model"] == model_id) & (scores["year"] == 2024)]
    wide = s.pivot_table(index="paper_id", columns="prompt_id", values="exp_score").reindex(papers["paper_id"])
    ax.vlines(xs, wide.min(axis=1), wide.max(axis=1), color=MUTED, linewidth=0.6, zorder=1)
    for p in PROMPTS:
        ax.scatter(xs, wide[p], s=9, marker=MARKERS[p], color=COLORS[p], linewidth=0, label=p, zorder=2)
    ax.plot(xs, papers["human_mean"], color=INK, linewidth=1.4, label="human mean rating", zorder=3)
    ax.set_ylim(1, 10)
    ax.set_yticks([2, 4, 6, 8, 10])
    ax.set_xticks([])
    ax.set_ylabel("Score (1–10)")
    ax.set_title(short(model_id), loc="left", fontsize=9)
axes[-1, 0].set_xlabel("Papers, sorted by mean human rating (low → high)")
axes[0, 0].legend(ncol=5, frameon=False, loc="lower left", bbox_to_anchor=(0, 1.15), fontsize=8)
fig.tight_layout()
save(fig, "fig1_scores_vs_human")

# Figure 2 (ICLR 2024): agreement per prompt with 95% CI, and the human ceiling band
yr = results["years"][2024]
hc = yr["human_ceiling"]
labels = list(PROMPTS) + ["average"]
fig, axes = plt.subplots(1, len(MODEL_IDS), figsize=(2.3 * len(MODEL_IDS), 2.4), sharex=True, sharey=True, squeeze=False)
for ax, model_id in zip(axes[0], MODEL_IDS):
    ax.axvspan(hc["human_ci"][0], hc["human_ci"][1], color=GRID, zorder=0)
    ax.axvline(hc["human_single_vs_others"], color=MUTED, linewidth=1, zorder=0)
    for yv, p in enumerate(labels):
        a = yr["models"][model_id]["agreement"][p]
        ax.plot(a["spearman_ci"], [yv, yv], color=MUTED, linewidth=1, zorder=1)
        ax.scatter(a["spearman"], yv, s=28, marker=MARKERS.get(p, "o"), color=COLORS.get(p, INK),
                   edgecolor="white", linewidth=0.6, zorder=2)
    ax.axvline(0, color=MUTED, linestyle="--", linewidth=0.8)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xlabel("Spearman ρ vs human")
    ax.set_title(short(model_id), loc="left", fontsize=9)
axes[0, 0].invert_yaxis()
fig.tight_layout()
save(fig, "fig2_agreement_vs_ceiling")

print(f"Done in {round((time.time() - RUN_START) / 60, 1)} min. Download data/ and results/ from the Output tab.")
if not os.path.isdir("/kaggle"):   # Colab / local: bundle everything into one file to download
    import zipfile
    with zipfile.ZipFile("study_v3_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for folder in ("data", "results"):
            for root, _, files in os.walk(f"{OUT}/{folder}"):
                for name in files:
                    z.write(os.path.join(root, name), os.path.relpath(os.path.join(root, name), OUT))
    print("Download study_v3_outputs.zip from the Files panel.")